### Treino de Transformers com Pythorch

Grande parte do código é o mesmo que o anterior, assim, focaremos a explicação na parte da arquitetura.

In [1]:
import pandas as pd
import numpy as np
import ast
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report

df = pd.read_csv('xrd_dataset_full.csv.gz', compression='gzip')

In [ ]:
def peaks_to_spectrum(peaks_str, grid_size=512, two_theta_range=(5, 90), sigma=0.15):
    try:
        peaks = ast.literal_eval(peaks_str)
    except:
        return np.zeros(grid_size)

    grid = np.linspace(two_theta_range[0], two_theta_range[1], grid_size)
    spectrum = np.zeros(grid_size)

    for peak in peaks:
        spectrum += np.exp(-(grid - peak)**2 / (2 * sigma**2))

    spectrum = spectrum / (spectrum.max() + 1e-8)
    return spectrum

X = np.stack(df['xray_peaks'].apply(peaks_to_spectrum).values)

X = (X - X.mean()) / (X.std() + 1e-8)

In [3]:
le = LabelEncoder()
y = le.fit_transform(df['crystal_system'])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [4]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

X_tr = torch.tensor(X_train, dtype=torch.float32).unsqueeze(-1)
X_te = torch.tensor(X_test, dtype=torch.float32).unsqueeze(-1)
y_tr = torch.tensor(y_train, dtype=torch.long)
y_te = torch.tensor(y_test, dtype=torch.long)

train_loader = DataLoader(TensorDataset(X_tr, y_tr), batch_size=64, shuffle=True)
test_loader = DataLoader(TensorDataset(X_te, y_te), batch_size=64)

In [5]:
weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
weights = torch.tensor(weights, dtype=torch.float32).to(device)
criterion = nn.CrossEntropyLoss(weight=weights)

In [ ]:
class PatchEmbedding1D(nn.Module):
    def __init__(self, seq_len=512, patch_size=8, d_model=128):
        super().__init__()
        self.proj = nn.Conv1d(
            in_channels=1,
            out_channels=d_model,
            kernel_size=patch_size,
            stride=patch_size
        )

    def forward(self, x):
        x = x.permute(0, 2, 1)
        x = self.proj(x)
        return x.permute(0, 2, 1)

Veja que, agora, as principais mudanças foram:

Uso de PatchEmbedding1D: agora o vetor de entrada é segmentado em patches (subsequências locais), que são projetados para um espaço de dimensão $d_{model}$ via convolução. Isso reduz o comprimento efetivo da sequência e introduz um viés indutivo de localidade.

CLS Token: um vetor treinável é adicionado no início da sequência de tokens e atua como um agregador global de informação. Durante o processo de self-attention, esse token aprende a “consultar” os demais tokens (patches), atribuindo pesos diferentes a cada região da entrada. Ao final das camadas do Transformer, o embedding associado ao CLS concentra uma representação global da amostra, que é então utilizada para classificação. Diferentemente do mean pooling, essa abordagem permite que o modelo aprenda dinamicamente quais partes da entrada são mais relevantes.

In [7]:
class XRDTransformer(nn.Module):
    def __init__(self, seq_len=512, patch_size=8,
                 d_model=128, nhead=4, num_layers=4,
                 num_classes=7):
        super().__init__()

        self.patch_embed = PatchEmbedding1D(seq_len, patch_size, d_model)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=512,
            dropout=0.1,
            batch_first=True
        )

        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers)

        self.cls_token = nn.Parameter(torch.randn(1, 1, d_model))
        self.classifier = nn.Linear(d_model, num_classes)

    def forward(self, x):
        x = self.patch_embed(x)

        B = x.size(0)
        cls_tokens = self.cls_token.expand(B, -1, -1)
        x = torch.cat([cls_tokens, x], dim=1)

        x = self.transformer(x)

        return self.classifier(x[:, 0])

model = XRDTransformer().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-2)

In [8]:
for epoch in range(100):
    model.train()
    total_loss = 0

    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)

        optimizer.zero_grad()
        out = model(xb)
        loss = criterion(out, yb)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1:02d} | Loss: {total_loss/len(train_loader):.4f}")

Epoch 01 | Loss: 1.1510


Epoch 02 | Loss: 0.9648


Epoch 03 | Loss: 0.9195


Epoch 04 | Loss: 0.8998


Epoch 05 | Loss: 0.8766


...

Epoch 96 | Loss: 0.5263


Epoch 97 | Loss: 0.5274


Epoch 98 | Loss: 0.5236


Epoch 99 | Loss: 0.5186


Epoch 100 | Loss: 0.5176


In [9]:
model.eval()
preds = []

with torch.no_grad():
    for xb, _ in test_loader:
        logits = model(xb.to(device))
        preds.append(logits.argmax(dim=1).cpu())

preds = torch.cat(preds)

acc = (preds == y_te).float().mean()
print(f"\nAcurácia: {acc:.4f}")

print("\nRelatório completo:")
print(classification_report(y_te, preds, target_names=le.classes_))


Acurácia: 0.6984

Relatório completo:
              precision    recall  f1-score   support

       cubic       0.97      1.00      0.98      2947
   hexagonal       0.39      0.68      0.49      2035
  monoclinic       0.40      0.52      0.45       306
orthorhombic       0.86      0.78      0.82      5470
  tetragonal       0.55      0.44      0.49      3266
   triclinic       0.83      0.90      0.86      1770
    trigonal       0.54      0.39      0.45      2435

    accuracy                           0.70     18229
   macro avg       0.65      0.67      0.65     18229
weighted avg       0.72      0.70      0.70     18229



Veja que com pouquíssimas modificações na estrutura do Transformer, foi possível obter grande melhora na acurácia, chegando a 70%. Além disso, é perceptível que o modelo aprendeu suficientemente bem cada estrutura, mesmo com o grande desbalanceamento. Assim, os Transformers se apresentam como ótima arquitetura para o meio acadêmico.